# Hugging Face 모델 탐색 실습

이번 실습에서는 Hugging Face에서 제공하는 다양한 인공지능 모델을 직접 찾아보고, Colab에서 실행해보겠습니다.

오늘은 모델을 직접 학습시키는 것이 아니라, 이미 학습되어 공개된 모델을 불러와 사용하는 방법을 연습합니다.

이번 시간에 실습할 task는 다음 4가지입니다.

1. 감정분석
2. 번역
3. 한국어 요약
4. 문장 유사도 비교


#라이브러리 설치

Hugging Face 모델을 사용하기 위해 필요한 라이브러리를 먼저 설치하겠습니다.

아래 코드는 실습에 필요한 기본 라이브러리를 설치하는 코드입니다.
처음 한 번만 실행하면 됩니다.

transformers: Hugging Face 모델을 불러오고 실행하는 라이브러리입니다.
sentencepiece: 일부 번역 및 요약 모델에서 사용하는 토크나이저 라이브러리입니다.
sentence-transformers: 문장을 벡터로 바꾸고 유사도를 계산할 때 사용하는 라이브러리입니다.

In [1]:
!pip install -q transformers sentencepiece sentence-transformers accelerate

In [2]:
import transformers
import sentence_transformers

print("transformers:", transformers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

transformers: 4.44.2
sentence-transformers: 5.6.0


#Hugging Face 모델 검색 방법

Hugging Face에는 다양한 인공지능 모델이 공개되어 있습니다.

이번 실습에서는 원하는 task에 맞는 모델을 직접 검색해보고, Colab에서 실행해보겠습니다.

모델을 찾는 순서는 다음과 같습니다.

1. Hugging Face 사이트에 접속합니다.
2. 상단 메뉴에서 `Models`를 클릭합니다.
3. 왼쪽 메뉴에서 원하는 `Task`를 선택합니다.
4. 검색창에 원하는 키워드를 입력합니다.
5. 마음에 드는 모델 페이지에 들어갑니다.
6. 모델 이름을 복사합니다.
7. Colab 코드의 `model_name` 부분에 붙여넣습니다.

모델 이름은 보통 다음과 같은 형태입니다.

```text
사용자명/모델명
```

예를 들어 다음과 같은 이름을 사용할 수 있습니다.

```text
distilbert-base-uncased-finetuned-sst-2-english
eenzeenee/t5-base-korean-summarization
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
```

처음에는 어떤 모델을 골라야 할지 헷갈릴 수 있습니다.
그럴 때는 다운로드 수가 많거나, 모델 카드에 사용 예시가 잘 적혀 있는 모델을 먼저 선택해보면 좋습니다.


#1. 감정분석
- sentiment-analysis
- text-classificatio

In [3]:
from transformers import pipeline

pipe = pipeline("text-classification", model="nlp04/korean_sentiment_analysis_kcelectra")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [4]:
output=pipe("와, 진짜 대단하다. 이런 식으로 망치기도 쉽지 않은데.")
print(output)

[{'label': '짜증남', 'score': 0.7652285695075989}]


In [5]:
pipe.model.config.id2label

{0: '기쁨(행복한)',
 1: '고마운',
 2: '설레는(기대하는)',
 3: '사랑하는',
 4: '즐거운(신나는)',
 5: '일상적인',
 6: '생각이 많은',
 7: '슬픔(우울한)',
 8: '힘듦(지침)',
 9: '짜증남',
 10: '걱정스러운(불안한)'}

#2. 번역
- translation

In [14]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# We will directly load the model and tokenizer, and define a translation function.

model_name = "facebook/nllb-200-distilled-600M"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def translate(text, direction="ko-en"):
    if direction == "ko-en":
        # NLLB models use language codes for source and target languages
        source_lang = "kor_Latn" # Korean
        target_lang = "eng_Latn" # English
    else:
        source_lang = "eng_Latn" # English
        target_lang = "kor_Latn" # Korean

    # Set the source language on the tokenizer object before tokenizing
    tokenizer.src_lang = source_lang

    inputs = tokenizer(
        text,
        return_tensors = "pt",
        max_length     = 512,
        truncation     = True,
    )
    outputs = model.generate(
        **inputs,
        # Corrected: Use tokenizer.convert_tokens_to_ids for the target language
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(target_lang),
        max_new_tokens     = 512,
        num_beams          = 4,
        early_stopping     = True,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Example usage:
print("Korean → English:")
print(translate("오늘 날씨가 정말 좋네요.", direction="ko-en"))

print("\nEnglish → Korean:")
print(translate("The weather is really nice today.", direction="en-ko"))

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Korean → English:
The weather is really nice today.

English → Korean:
Today the weather is really nice.


In [10]:
# Korean → English
print(translate("가격만 아니었으면 완벽했다.", direction="ko-en"))

# English → Korean
print(translate("We need transformer to make a transformer", direction="en-ko"))

ILLARShiftShiftShift 腾讯时时彩 腾讯时时彩 腾讯时时彩 腾讯时时彩-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-000000000000","-00000000000

KeyboardInterrupt: 

#3. 요약
- summarization
- text2text-generation

In [15]:
!pip install -q transformers tokenizers sentencepiece accelerate

In [19]:
import nltk
nltk.download('punkt') # Ensure punkt tokenizer is downloaded
nltk.download('punkt_tab') # Download punkt_tab for sentence tokenization
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# The task 'summarization' is not recognized in the current transformers pipeline.
# To fix this, we will directly load the model and tokenizer for summarization.

model_name = "eenzeenee/t5-base-korean-summarization"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

def summarize_text(text, max_length=64, min_length=10, num_beams=3):
    prefix = "summarize: "
    inputs = [prefix + text]

    inputs = tokenizer(inputs, max_length=512, truncation=True, return_tensors="pt")
    output = model.generate(**inputs, num_beams=num_beams, do_sample=True, min_length=min_length, max_length=max_length)
    decoded_output = tokenizer.batch_decode(output, skip_special_tokens=True)[0]

    # Use nltk.sent_tokenize after ensuring 'punkt' is downloaded
    result = nltk.sent_tokenize(decoded_output.strip())[0]
    return result

# Example usage:
sample_text = """
    안녕하세요? 우리 (2학년)/(이 학년) 친구들 우리 친구들 학교에 가서 진짜 (2학년)/(이 학년) 이 되고 싶었는데 학교에 못 가고 있어서 답답하죠?
    그래도 우리 친구들의 안전과 건강이 최우선이니까요 오늘부터 선생님이랑 매일 매일 국어 여행을 떠나보도록 해요.
    어/ 시간이 벌써 이렇게 됐나요? 늦었어요. 늦었어요. 빨리 국어 여행을 떠나야 돼요.
    그런데 어/ 국어여행을 떠나기 전에 우리가 준비물을 챙겨야 되겠죠? 국어 여행을 떠날 준비물, 교안을 어떻게 받을 수 있는지 선생님이 설명을 해줄게요.
    (EBS)/(이비에스) 초등을 검색해서 들어가면요 첫화면이 이렇게 나와요.
    자/ 그러면요 여기 (X)/(엑스) 눌러주(고요)/(구요). 저기 (동그라미)/(똥그라미) (EBS)/(이비에스) (2주)/(이 주) 라이브특강이라고 되어있죠?
    거기를 바로 가기를 누릅니다. 자/ (누르면요)/(눌르면요). 어떻게 되냐? b/ 밑으로 내려요 내려요 내려요 쭉 내려요.
    우리 몇 학년이죠? 아/ (2학년)/(이 학년) 이죠 (2학년)/(이 학년)의 무슨 과목? 국어.
    이번주는 (1주)/(일 주) 차니까요 여기 교안. 다음주는 여기서 다운을 받으면 돼요.
    이 교안을 클릭을 하면, 짜잔/. 이렇게 교재가 나옵니다 .이 교안을 (다운)/(따운)받아서 우리 국어여행을 떠날 수가 있어요.
    그럼 우리 진짜로 국어 여행을 한번 떠나보도록 해요? 국어여행 출발. 자/ (1단원)/(일 단원) 제목이 뭔가요? 한번 찾아봐요.
    시를 즐겨요 에요. 그냥 시를 읽어요 가 아니에요. 시를 즐겨야 돼요 즐겨야 돼. 어떻게 즐길까? 일단은 내내 시를 즐기는 방법에 대해서 공부를 할 건데요.
    그럼 오늘은요 어떻게 즐길까요? 오늘 공부할 내용은요 시를 여러 가지 방법으로 읽기를 공부할겁니다.
    어떻게 여러가지 방법으로 읽을까 우리 공부해 보도록 해요. 오늘의 시 나와라 짜잔/! 시가 나왔습니다 시의 제목이 뭔가요? 다툰 날이에요 다툰 날.
    누구랑 다퉜나 동생이랑 다퉜나 언니랑 친구랑? 누구랑 다퉜는지 선생님이 시를 읽어 줄 테니까 한번 생각을 해보도록 해요."""

print('Summary >>', summarize_text(sample_text))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Summary >> 국어 여행을 떠나기 전에 국어 여행을 떠날 준비물과 교안을 어떻게 받을 수 있는지 선생님이 설명해 준다.


In [20]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab') # Download punkt_tab for sentence tokenization
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained('eenzeenee/t5-base-korean-summarization')
tokenizer = AutoTokenizer.from_pretrained('eenzeenee/t5-base-korean-summarization')

prefix = "summarize: "
sample = """
    안녕하세요? 우리 (2학년)/(이 학년) 친구들 우리 친구들 학교에 가서 진짜 (2학년)/(이 학년) 이 되고 싶었는데 학교에 못 가고 있어서 답답하죠?
    그래도 우리 친구들의 안전과 건강이 최우선이니까요 오늘부터 선생님이랑 매일 매일 국어 여행을 떠나보도록 해요.
    어/ 시간이 벌써 이렇게 됐나요? 늦었어요. 늦었어요. 빨리 국어 여행을 떠나야 돼요.
    그런데 어/ 국어여행을 떠나기 전에 우리가 준비물을 챙겨야 되겠죠? 국어 여행을 떠날 준비물, 교안을 어떻게 받을 수 있는지 선생님이 설명을 해줄게요.
    (EBS)/(이비에스) 초등을 검색해서 들어가면요 첫화면이 이렇게 나와요.
    자/ 그러면요 여기 (X)/(엑스) 눌러주(고요)/(구요). 저기 (동그라미)/(똥그라미) (EBS)/(이비에스) (2주)/(이 주) 라이브특강이라고 되어있죠?
    거기를 바로 가기를 누릅니다. 자/ (누르면요)/(눌르면요). 어떻게 되냐? b/ 밑으로 내려요 내려요 내려요 쭉 내려요.
    우리 몇 학년이죠? 아/ (2학년)/(이 학년) 이죠 (2학년)/(이 학년)의 무슨 과목? 국어.
    이번주는 (1주)/(일 주) 차니까요 여기 교안. 다음주는 여기서 다운을 받으면 돼요.
    이 교안을 클릭을 하면, 짜잔/. 이렇게 교재가 나옵니다 .이 교안을 (다운)/(따운)받아서 우리 국어여행을 떠날 수가 있어요.
    그럼 우리 진짜로 국어 여행을 한번 떠나보도록 해요? 국어여행 출발. 자/ (1단원)/(일 단원) 제목이 뭔가요? 한번 찾아봐요.
    시를 즐겨요 에요. 그냥 시를 읽어요 가 아니에요. 시를 즐겨야 돼요 즐겨야 돼. 어떻게 즐길까? 일단은 내내 시를 즐기는 방법에 대해서 공부를 할 건데요.
    그럼 오늘은요 어떻게 즐길까요? 오늘 공부할 내용은요 시를 여러 가지 방법으로 읽기를 공부할겁니다.
    어떻게 여러가지 방법으로 읽을까 우리 공부해 보도록 해요. 오늘의 시 나와라 짜잔/! 시가 나왔습니다 시의 제목이 뭔가요? 다툰 날이에요 다툰 날.
    누구랑 다퉜나 동생이랑 다퉜나 언니랑 친구랑? 누구랑 다퉜는지 선생님이 시를 읽어 줄 테니까 한번 생각을 해보도록 해요."""

inputs = [prefix + sample]


inputs = tokenizer(inputs, max_length=512, truncation=True, return_tensors="pt")
output = model.generate(**inputs, num_beams=3, do_sample=True, min_length=10, max_length=64)
decoded_output = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
result = nltk.sent_tokenize(decoded_output.strip())[0]

print('RESULT >>', result)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


RESULT >> 국어 여행을 떠나기 전에 국어 여행을 떠날 준비물과 교안을 어떻게 받을 수 있는지 선생님이 설명해 준다.


#4. 문장 유사도 비교
- sentence-similarity
- sentence-transformers 라이브러리 사용

In [21]:
from sentence_transformers import SentenceTransformer, util

# Load a pre-trained multilingual model
# Using 'paraphrase-multilingual-MiniLM-L12-v2' as it is a commonly used and publicly available alternative.
sentence_model_name = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
sentence_model = SentenceTransformer(sentence_model_name)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [22]:
sentences = [
    "오늘 날씨가 정말 좋네요.",
    "날씨가 오늘 정말 화창합니다.",
    "저는 오늘 콜라를 마셨습니다.",
    "The weather is beautiful today."
]

# Encode all sentences into embeddings using the sentence_model
embeddings = sentence_model.encode(sentences, convert_to_tensor=True)

# Compute cosine similarity between all pairs
cosine_scores = util.cos_sim(embeddings, embeddings)

print("Sentence Similarity:")
for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        print(f"Sentence 1: '{sentences[i]}'")
        print(f"Sentence 2: '{sentences[j]}'")
        print(f"Similarity Score: {cosine_scores[i][j].item():.4f}\n")

Sentence Similarity:
Sentence 1: '오늘 날씨가 정말 좋네요.'
Sentence 2: '날씨가 오늘 정말 화창합니다.'
Similarity Score: 0.8675

Sentence 1: '오늘 날씨가 정말 좋네요.'
Sentence 2: '저는 오늘 콜라를 마셨습니다.'
Similarity Score: 0.2809

Sentence 1: '오늘 날씨가 정말 좋네요.'
Sentence 2: 'The weather is beautiful today.'
Similarity Score: 0.8138

Sentence 1: '날씨가 오늘 정말 화창합니다.'
Sentence 2: '저는 오늘 콜라를 마셨습니다.'
Similarity Score: 0.2782

Sentence 1: '날씨가 오늘 정말 화창합니다.'
Sentence 2: 'The weather is beautiful today.'
Similarity Score: 0.8150

Sentence 1: '저는 오늘 콜라를 마셨습니다.'
Sentence 2: 'The weather is beautiful today.'
Similarity Score: 0.1618

